[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L11.ipynb)

# Lecture 11 — From Scratch & nn.Module

*Build the whole thing with your bare hands once — then meet the class that does it properly.*

**PyTorch from Ground Up · Volume I · Chapters 18–19** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Train a model using nothing but raw tensors, autograd and arithmetic.
- Explain why the manual update is wrapped in `torch.no_grad()`.
- Say exactly what the optimizer and `nn.Linear` were doing for you.
- Write a model as an `nn.Module` with `__init__` and `forward`.
- Use `.parameters()`, `named_parameters()` and `print(model)` to inspect a model.
- Move a model — and its data — to a device.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · Why build it by hand?

Before using the tools properly, build the model **with your bare hands**. Nobody works this way in practice. But once you have, `nn.Module` and the optimizer will never feel like magic again.

<sub>Source: lecture slides 3–8</sub>

**`setup.py`** — predict the output, then run.

In [ ]:
import torch
torch.manual_seed(0)
# Data — same as Chapter 3
x = torch.linspace(0, 2, 20).unsqueeze(1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)

# Our "model": y_hat = w * x + b
w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
print(f"Initial: w={w.item():.4f}, b={b.item():.4f}")

<details><summary>▸ Expected output</summary>

```
Initial: w=-0.8920, b=0.0000
```
</details>

## 02 · The fully manual training loop

Those two subtraction lines **are `optimizer.step()`** written out longhand. Same five-step rhythm — but a formula for the forward pass, MSE by hand, and raw arithmetic for the update. It still learned the same best-fit line, 3.29x + 0.82.

<sub>Source: lecture slides 9–13, 19–20</sub>

**`training_loop.py`** — predict the output, then run.

In [ ]:
lr = 0.1
for epoch in range(200):
    # ❶ Forward pass — no model object
    y_hat = w * x + b
    # ❷ Loss — MSE by hand
    loss = ((y_hat - y) ** 2).mean()
    # ❸ Backward — autograd
    loss.backward()
    # ❹ Update — manual gradient descent
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad
    # ❺ Zero gradients for next step
    w.grad.zero_()
    b.grad.zero_()
    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d}   "
              f"Loss: {loss.item():.4f}   "
              f"w={w.item():.3f} b={b.item():.3f}")

<details><summary>▸ Expected output</summary>

```
Epoch   0   Loss: 31.4709   w=0.415 b=0.999
Epoch  50   Loss: 0.0900   w=3.059 b=1.092
Epoch 100   Loss: 0.0674   w=3.248 b=0.865
Epoch 150   Loss: 0.0667   w=3.282 b=0.823
```
</details>

🧪 *Interactive lab in the app:* **Delete a safety line** — try it there to see this idea move.

## 03 · Why no_grad wraps the update

The update itself touches a tracked tensor — so **keep it off the tape**. Without `no_grad`, PyTorch would record `w -= lr * w.grad` in the graph. Wrapping it says: bookkeeping, not computation.

<sub>Source: lecture slides 14–15</sub>

> ⚠️ **Watch out — Forget no_grad and it breaks**  
> Without the wrapper, PyTorch raises immediately — on the first epoch:
>
> `RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.`
>
> `-=` mutates the tensor in place, and autograd refuses to let a tracked leaf tensor change underneath a graph it may still need. `optimizer.step()` handles all of this for you automatically.

## 04 · What the optimizer and nn.Linear were doing for you

The optimizer is **packaging** — nothing you can’t do by hand. It bundles the update, the zeroing, the `no_grad` wrapper and the algorithm (momentum, Adam). For two parameters, a convenience; for millions, a necessity.

<sub>Source: lecture slides 16–18, 22–23</sub>

## 05 · nn.Module: the two-method pattern

The manual model worked — and **could never scale** to fifty layers. `nn.Module` can. It brings parameter tracking, a clean `forward`, device moves, saving and loading, and modules nested in modules. Every model from here on subclasses it.

<sub>Source: lecture slides 24–32</sub>

**`model.py`** — predict the output, then run.

In [ ]:
import torch
import torch.nn as nn

class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)

> ⚠️ **Watch out — Always call model(x), never model.forward(x)**  
> Calling the model directly runs extra hooks — for things like dropout and batch normalization — that `forward` alone would skip.
>
> It will not error. It will just quietly do the wrong thing.

**`train.py`** — predict the output, then run.

In [ ]:
torch.manual_seed(0)
x = torch.linspace(0, 2, 20).unsqueeze(1)
y = 3 * x + 1 + 0.3 * torch.randn_like(x)
model = TinyModel()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# Training loop — unchanged
for epoch in range(200):
    loss = loss_fn(model(x), y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
w = model.linear.weight.item()
b = model.linear.bias.item()
print(f"y ≈ {w:.2f}x + {b:.2f}")

<details><summary>▸ Expected output</summary>

```
y ≈ 3.20x + 0.92
```
</details>

## 06 · What the optimizer sees: .parameters()

`model.parameters()` walks your `__init__` and **collects every learnable tensor** — recursively. Exactly the bookkeeping you did by hand, now automatic, through every nested sub-module.

<sub>Source: lecture slides 33–41</sub>

**`inspect.py`** — predict the output, then run.

In [ ]:
for name, param in model.named_parameters():
    print(name, param.shape)

<details><summary>▸ Expected output</summary>

```
linear.weight torch.Size([1, 1])
linear.bias torch.Size([1])
```
</details>

**`two_layer.py`** — predict the output, then run.

In [ ]:
class TwoLayer(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.layer1 = nn.Linear(in_dim, hidden)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

model = TwoLayer(1, 16, 1)
print(sum(p.numel() for p in model.parameters()))
print(model)

<details><summary>▸ Expected output</summary>

```
49
TwoLayer(
  (layer1): Linear(in_features=1, out_features=16, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=16, out_features=1, bias=True)
)
```
</details>

🧪 *Interactive lab in the app:* **Count the parameters** — try it there to see this idea move.

## 07 · Moving to a device

One call moves **every parameter**, even deep inside nested modules.

<sub>Source: lecture slides 42–44</sub>

**`device.py`** — predict the output, then run.

In [ ]:
device = "cuda" if torch.cuda.is_available() \
    else "cpu"
model = model.to(device)     # every parameter moves

> ⚠️ **Watch out — The data has to move too**  
> After `model.to(device)`, your data must also be on the same device — `x = x.to(device)`.
>
> Mixing a CPU model with GPU data (or vice versa) is a runtime error — one of the most common mistakes when first using a GPU.

## The rule — worth remembering

**From scratch — the five rules**

1. A model is just parameters + a forward function. You can build one from raw tensors and arithmetic.
2. Set `requires_grad=True` on your parameters so autograd tracks them.
3. Wrap the manual update in `torch.no_grad()` to keep it out of the computation graph.
4. Zero the gradients after every update — they accumulate by default.
5. `nn.Module` and the optimizer do exactly this, but cleaner, safer, and scalable to millions of parameters.

**nn.Module — the six rules**

1. Inherit from `nn.Module`. Call `super().__init__()` first in `__init__`.
2. Create layers and sub-modules as attributes in `__init__` so they are registered.
3. Define the data flow in `forward`. Call the model with `model(x)`, never `model.forward(x)`.
4. `.parameters()` collects every learnable tensor — recursively through all sub-modules.
5. `.to(device)` moves every parameter at once.
6. Printing the model shows its full architecture for free.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Try it — two features

Extend the manual model to two inputs: y = 3x₁ + 2x₂ + 1. Create `w` as a (1, 2) tensor and `x` as (20, 2). Modify the forward pass to use `x @ w.T + b`. Train it and confirm the learned weights are close to [3, 2] and the bias close to 1.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`x = torch.rand(20, 2) * 2; y = 3*x[:, :1] + 2*x[:, 1:] + 1 + 0.3*torch.randn(20, 1)`; `w = torch.randn(1, 2, requires_grad=True)`; forward `y_hat = x @ w.T + b` gives (20, 1). The rest of the loop is unchanged, and w converges near [[3, 2]] with b near 1.
</details>

### Exercise 02 · Ch 18 · Ex 2

Remove the `torch.no_grad()` wrapper from the update and train again. What happens, and why?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`, on the first epoch — the in-place `-=` would change a tracked leaf tensor underneath the graph. Untick the line in the lab to see it.
</details>

### Exercise 03 · Ch 18 · Ex 5

Compare the final loss of the manual model to the Chapter 3 `nn.Linear` model. Are they the same? Should they be?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

Both end at loss 0.0667 and recover y ≈ 3.29x + 0.82, because both run plain gradient descent on the same data toward the same minimum. The epoch-0 losses differ (31.5 vs 20.3) only because the starting weights differ — and they should converge to the same answer.
</details>

### Exercise 04 · Ch 19 · Ex 1

Write a `ThreeLayer` model with two hidden layers of size 32 and 16, ReLU between each. Print the model and count its parameters.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

For input and output size 1: (1·32 + 32) + (32·16 + 16) + (16·1 + 1) = 64 + 528 + 17 = **609** parameters.
</details>

### Exercise 05 · Ch 19 · Ex 4

Store a layer as a local variable inside `forward` instead of an attribute in `__init__`. Does `model.parameters()` find it? Why or why not?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

No. Registration happens when a module is assigned as an attribute (`self.layer = …`). A local variable is invisible to `.parameters()` — and it is also re-created with fresh random weights on every forward call, so it could never learn.
</details>

### Interview-style

(a) “Walk me through what happens inside `optimizer.step()`,” naming two things it handles that a manual loop doesn’t.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

For every parameter it was given, `step()` reads the gradient stored in `.grad` and applies the update rule — `param -= lr * grad` for plain SGD, or the momentum/Adam variant. It does this without tracking the update in the computation graph (the `no_grad` wrapper you would otherwise write yourself), and it loops over every parameter automatically instead of one hand-written line each. It also carries the algorithm’s state, such as momentum buffers.
</details>

### Interview-style

(b) “What does `nn.Module` give you that raw tensors don’t?” — three concrete things, plus why `model(x)` beats `model.forward(x)`.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Automatic parameter registration and discovery through `.parameters()`, even in nested sub-modules; one-call device movement with `.to(device)`; and saving/loading plus a readable `print(model)`. Calling `model(x)` runs the module’s hooks around `forward`, which calling `forward` directly would silently skip.
</details>